In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. 충돌 가능성 있는 패키지 전부 제거
!pip uninstall -y torch torchvision torchaudio transformers tokenizers accelerate numpy numba opencv-python opencv-python-headless peft sentence-transformers

# 2. numpy는 반드시 1.x 버전으로 고정
!pip install numpy==1.26.4

# 3. PyTorch 안정 조합 (CUDA 12.1 기준, Colab GPU 호환)
!pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121

# 4. HuggingFace 관련 패키지 (버전 호환 고정)
!pip install transformers==4.35.2 tokenizers==0.15.2 accelerate==0.21.0 datasets==2.16.1

# (선택) sentence-transformers가 필요하면 호환되는 버전으로
!pip install sentence-transformers==2.2.2


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from collections import Counter

# 데이터 로딩
df = pd.read_csv('/content/drive/MyDrive/감정분석모델/감정분석_70감정_혼합표현_최종본_UTF8SIG.csv')
df = df.dropna()

# 라벨 인코딩
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['감정명'])

# 라벨 분포 확인 (클래스 불균형 확인용)
print("감정 라벨 분포:", Counter(df['label']))

df.head()

감정 라벨 분포: Counter({46: 30, 28: 30, 36: 30, 44: 30, 19: 30, 26: 30, 56: 30, 55: 30, 42: 30, 53: 30, 38: 30, 65: 30, 35: 30, 48: 30, 51: 30, 57: 30, 61: 30, 54: 30, 64: 30, 67: 30, 12: 30, 31: 30, 34: 30, 66: 30, 5: 30, 6: 30, 62: 30, 22: 30, 24: 30, 21: 30, 18: 30, 20: 30, 4: 30, 45: 30, 8: 30, 52: 30, 59: 30, 1: 30, 2: 30, 27: 30, 14: 30, 29: 30, 17: 30, 32: 30, 41: 30, 11: 30, 3: 30, 25: 30, 23: 30, 10: 30, 33: 30, 40: 30, 63: 30, 60: 30, 16: 30, 58: 30, 68: 30, 43: 30, 37: 30, 30: 30, 9: 30, 47: 30, 0: 30, 39: 30, 49: 30, 69: 30, 50: 30, 13: 30, 15: 30, 7: 30})


,감정명,문장 예시,label
0,우울감,가끔은 우울감 없이 살고 싶다는 생각이 들어.,46
1,우울감,그냥 멍하게 있다 보면 우울감이 밀려와.,46
2,우울감,우울감이 쌓이고 쌓여서 이제는 무뎌진 것 같아.,46
3,우울감,사람들이 모를 뿐이지 나 진짜 우울감해.,46
4,우울감,진심으로 우울감이 나를 갉아먹는 느낌이야.,46


In [ ]:
from sklearn.model_selection import train_test_split

# 학습용 문장과 라벨 분리
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['문장 예시'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
num_labels = len(label_encoder.classes_)
print(f"총 감정 클래스 수: {num_labels}")

총 감정 클래스 수: 70


In [ ]:
from transformers import AutoTokenizer
import torch

# ✅ 최신 KoBERT 모델 경로 (Hugging Face 공식)
model_name = "skt/kobert-base-v1"

# ✅ KoBERT 전용 tokenizer (AutoTokenizer로 자동 처리)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ 입력 최대 길이
max_len = 64

# ✅ 토크나이즈
train_encodings = tokenizer(list(train_texts), truncation=True, padding='max_length', max_length=max_len)
val_encodings = tokenizer(list(val_texts), truncation=True, padding='max_length', max_length=max_len)

# ✅ Dataset 클래스 정의
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

# ✅ Dataset 객체 생성
train_dataset = EmotionDataset(train_encodings, list(train_labels))
val_dataset = EmotionDataset(val_encodings, list(val_labels))



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.11/dist-package

In [ ]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['감정명'])

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "skt/kobert-base-v1",  # ✅ 최신 KoBERT 경로
    num_labels=len(label_encoder.classes_),
    id2label={i: label for i, label in enumerate(label_encoder.classes_)},
    label2id={label: i for i, label in enumerate(label_encoder.classes_)}
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
import numpy as np

# classification_report를 출력형 문자열로 저장하고 싶다면:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    precision = precision_score(labels, preds, average="weighted", zero_division=0)
    recall = recall_score(labels, preds, average="weighted", zero_division=0)

    # report_str = classification_report(labels, preds, target_names=label_encoder.classes_)
    # print(report_str)  # 필요시 출력 가능

    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,  # Reduced batch size
    per_device_eval_batch_size=8,   # Reduced batch size
    num_train_epochs=5,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=10,
    report_to='none'
)

In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,  # ✅ tokenizer도 잘 연동됨
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # ✅ 조기 종료 콜백 추가
)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("skt/kobert-base-v1")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
from torch.utils.data import Dataset

MAX_LEN = 64  # 그대로 사용 가능

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        # Ensure label is an integer and within the valid range
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long) # Explicitly cast to torch.long
        }

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from collections import Counter

# ✅ CSV 경로 (한글 경로 인코딩 문제 없도록 주의)
csv_path = "/content/drive/MyDrive/감정분석모델/감정분석_70감정_혼합표현_최종본_UTF8SIG.csv"

# ✅ 데이터 로딩
df = pd.read_csv(csv_path)

# ✅ 결측치 제거 (중요!)
df = df.dropna(subset=['문장 예시', '감정명'])

# ✅ 문장과 라벨 분리
texts = df['문장 예시'].tolist()
labels = df['감정명'].tolist()

# ✅ 라벨 인코딩
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

# ✅ 클래스 수 및 불균형 확인
print(f"총 감정 클래스 수: {len(label_encoder.classes_)}")
print("라벨 분포:", Counter(encoded_labels))

총 감정 클래스 수: 70
라벨 분포: Counter({np.int64(46): 30, np.int64(28): 30, np.int64(36): 30, np.int64(44): 30, np.int64(19): 30, np.int64(26): 30, np.int64(56): 30, np.int64(55): 30, np.int64(42): 30, np.int64(53): 30, np.int64(38): 30, np.int64(65): 30, np.int64(35): 30, np.int64(48): 30, np.int64(51): 30, np.int64(57): 30, np.int64(61): 30, np.int64(54): 30, np.int64(64): 30, np.int64(67): 30, np.int64(12): 30, np.int64(31): 30, np.int64(34): 30, np.int64(66): 30, np.int64(5): 30, np.int64(6): 30, np.int64(62): 30, np.int64(22): 30, np.int64(24): 30, np.int64(21): 30, np.int64(18): 30, np.int64(20): 30, np.int64(4): 30, np.int64(45): 30, np.int64(8): 30, np.int64(52): 30, np.int64(59): 30, np.int64(1): 30, np.int64(2): 30, np.int64(27): 30, np.int64(14): 30, np.int64(29): 30, np.int64(17): 30, np.int64(32): 30, np.int64(41): 30, np.int64(11): 30, np.int64(3): 30, np.int64(25): 30, np.int64(23): 30, np.int64(10): 30, np.int64(33): 30, np.int64(40): 30, np.int64(63): 30, np.int64(60): 30, np.i

In [ ]:
from sklearn.model_selection import train_test_split

# ✅ 학습/검증 데이터 분할 (8:2 비율, 재현 가능하도록 시드 고정)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, encoded_labels, test_size=0.2, random_state=42, stratify=encoded_labels
)

# ✅ Dataset 인스턴스 생성 (EmotionDataset은 앞서 정의한 KoBERT용 클래스 사용)
train_dataset = EmotionDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = EmotionDataset(val_texts, val_labels, tokenizer, MAX_LEN)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "skt/kobert-base-v1",
    num_labels=len(label_encoder.classes_),
    id2label={i: label for i, label in enumerate(label_encoder.classes_)},
    label2id={label: i for i, label in enumerate(label_encoder.classes_)}
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

from transformers import DataCollatorWithPadding

# Add a hook to inspect model inputs and outputs
def debug_hook(module, inputs, outputs):
    print("Module:", type(module).__name__)
    print("Input types:", [i.dtype for i in inputs if hasattr(i, 'dtype')])
    print("Input shapes:", [i.shape for i in inputs if hasattr(i, 'shape')])
    # print("Output types:", [o.dtype for o in outputs if hasattr(o, 'dtype')]) # Uncomment if needed
    # print("Output shapes:", [o.shape for o in outputs if hasattr(o, 'shape')]) # Uncomment if needed


# Attach the hook to the entire model
model.register_forward_hook(debug_hook)
print("Debug hook registered successfully.")

# Initialize DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,  # Explicitly use DataCollatorWithPadding
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Debug hook registered successfully.


RuntimeError: Numpy is not available

In [ ]:
# 저장 경로 설정
save_path = "/content/drive/MyDrive/감정분석모델/klue-roberta-70emotion-v3"

# 모델 저장
trainer.save_model(save_path)

# 토크나이저 저장
tokenizer.save_pretrained(save_path)